Ноутбук считает метрики на InfoSearch. Это внешняя проверка instruction-following поведения модели на готовом benchmark-е.


In [ ]:
!pip uninstall -y bitsandbytes peft -q
!pip install -q peft==0.13.2 bitsandbytes==0.46.1 sentencepiece protobuf accelerate datasets faiss-cpu

import os, json, math, time, gc
from collections import defaultdict, Counter

import numpy as np
import torch
import torch.nn.functional as F
import faiss
from tqdm.auto import tqdm

from huggingface_hub import login, hf_hub_download, snapshot_download
from transformers import AutoTokenizer, AutoModel, BitsAndBytesConfig
from peft import PeftModel


TASKS = [
    "Audience-v1",
    "Format-v1",
    "Keyword-v1",
    "Language-v1",
    "Length-v1",
    "Source-v1",
]

REPO_ID = "EIT-NLP/InfoSearch"

BASE_MODEL = "meta-llama/Llama-2-7b-hf"
ORIGINAL_ADAPTER_REPO = "samaya-ai/promptriever-llama2-7b-v1"
FINETUNED_ADAPTER = "/kaggle/input/datasets/sukiss/adapters"

WORK_DIR = "/kaggle/working/infosearch_all_eval"
DATA_DIR = os.path.join(WORK_DIR, "data")
ADAPTER_CACHE = os.path.join(WORK_DIR, "adapter_cache")

os.makedirs(WORK_DIR, exist_ok=True)
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(ADAPTER_CACHE, exist_ok=True)

CORPUS_PATH = os.path.join(DATA_DIR, "infosearch_all_corpus.jsonl")
QUERIES_PATH = os.path.join(DATA_DIR, "infosearch_all_queries.jsonl")

QUERY_MAX_LEN = 192
PASSAGE_MAX_LEN = 128
BATCH_DOCS = 16
BATCH_QUERIES = 32

HF_TOKEN = globals().get("HF_TOKEN", os.environ.get("HF_TOKEN", ""))
if HF_TOKEN:
    login(token=HF_TOKEN) if HF_TOKEN else None

assert os.path.exists(FINETUNED_ADAPTER), FINETUNED_ADAPTER


def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

def qrels_to_map(rows):
    m = defaultdict(dict)
    for r in rows:
        qid = str(r["query-id"])
        did = str(r["corpus-id"])
        score = int(r["score"])
        m[qid][did] = score
    return m

def build_infosearch_all():
    if os.path.exists(CORPUS_PATH) and os.path.exists(QUERIES_PATH):
        print("Normalized InfoSearch already exists:")
        print(CORPUS_PATH)
        print(QUERIES_PATH)
        return

    all_corpus = []
    all_queries = []
    global_stats = Counter()
    task_stats = {}

    for task in TASKS:
        print("\nPreparing task:", task)

        paths = {
            "queries": hf_hub_download(
                repo_id=REPO_ID,
                repo_type="dataset",
                filename=f"{task}/queries.jsonl",
                token=HF_TOKEN if HF_TOKEN else None,
            ),
            "corpus": hf_hub_download(
                repo_id=REPO_ID,
                repo_type="dataset",
                filename=f"{task}/corpus.jsonl",
                token=HF_TOKEN if HF_TOKEN else None,
            ),
            "qrels_og": hf_hub_download(
                repo_id=REPO_ID,
                repo_type="dataset",
                filename=f"{task}/qrels_og/test.jsonl",
                token=HF_TOKEN if HF_TOKEN else None,
            ),
            "qrels_changed": hf_hub_download(
                repo_id=REPO_ID,
                repo_type="dataset",
                filename=f"{task}/qrels_changed/test.jsonl",
                token=HF_TOKEN if HF_TOKEN else None,
            ),
            "qrels_reversed": hf_hub_download(
                repo_id=REPO_ID,
                repo_type="dataset",
                filename=f"{task}/qrels_reversed/test.jsonl",
                token=HF_TOKEN if HF_TOKEN else None,
            ),
        }

        queries_raw = load_jsonl(paths["queries"])
        corpus_raw = load_jsonl(paths["corpus"])
        qrels_og = qrels_to_map(load_jsonl(paths["qrels_og"]))
        qrels_changed = qrels_to_map(load_jsonl(paths["qrels_changed"]))
        qrels_reversed = qrels_to_map(load_jsonl(paths["qrels_reversed"]))

        for d in corpus_raw:
            old_docid = str(d["_id"])
            new_docid = f"{task}::{old_docid}"
            all_corpus.append({
                "docid": new_docid,
                "task": task,
                "title": d.get("title", ""),
                "text": d.get("text", ""),
            })

        local_stats = Counter()

        for q in queries_raw:
            qid = str(q["_id"])
            new_qid = f"{task}::{qid}"

            og = qrels_og.get(qid, {})
            ch = qrels_changed.get(qid, {})
            rev = qrels_reversed.get(qid, {})

            all_docs = set(og) | set(ch) | set(rev)

            rel_og = sorted([f"{task}::{d}" for d in all_docs if og.get(d, 0) == 1])
            rel_changed = sorted([f"{task}::{d}" for d in all_docs if ch.get(d, 0) == 1])
            rel_reversed = sorted([f"{task}::{d}" for d in all_docs if rev.get(d, 0) == 1])

            target_docs = sorted([
                f"{task}::{d}" for d in all_docs
                if ch.get(d, 0) == 1 and rev.get(d, 0) == 0
            ])

            pmrr_changed_docs = sorted([
                f"{task}::{d}" for d in all_docs
                if og.get(d, 0) == 1 and ch.get(d, 0) == 0
            ])

            if not rel_og:
                local_stats["skip_no_rel_og"] += 1
                continue
            if not rel_changed:
                local_stats["skip_no_rel_changed"] += 1
                continue
            if not rel_reversed:
                local_stats["skip_no_rel_reversed"] += 1
                continue
            if not target_docs:
                local_stats["skip_no_target_docs"] += 1
                continue
            if not pmrr_changed_docs:
                local_stats["skip_no_pmrr_changed_docs"] += 1
                continue

            all_queries.append({
                "qid": new_qid,
                "task": task,
                "query": q.get("text", "").strip(),
                "instruction_og": q.get("instruction_og", "").strip(),
                "instruction_changed": q.get("instruction_changed", "").strip(),
                "instruction_reversed": q.get("instruction_reversed", "").strip(),
                "short_query": q.get("short_query", "").strip(),
                "keywords": q.get("keywords", ""),

                "rel_og": rel_og,
                "rel_changed": rel_changed,
                "rel_reversed": rel_reversed,

                "target_docs": target_docs,
                "pmrr_changed_docs": pmrr_changed_docs,
            })

            local_stats["kept"] += 1
            local_stats["rel_og_total"] += len(rel_og)
            local_stats["rel_changed_total"] += len(rel_changed)
            local_stats["rel_reversed_total"] += len(rel_reversed)
            local_stats["target_docs_total"] += len(target_docs)
            local_stats["pmrr_changed_docs_total"] += len(pmrr_changed_docs)

        task_stats[task] = dict(local_stats)
        global_stats.update(local_stats)

    with open(CORPUS_PATH, "w", encoding="utf-8") as f:
        for x in all_corpus:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")

    with open(QUERIES_PATH, "w", encoding="utf-8") as f:
        for x in all_queries:
            f.write(json.dumps(x, ensure_ascii=False) + "\n")

    print("\nSaved corpus:", CORPUS_PATH)
    print("Saved queries:", QUERIES_PATH)
    print("Corpus docs:", len(all_corpus))
    print("Queries:", len(all_queries))
    print("Global stats:", global_stats)
    print("Task stats:")
    for task, st in task_stats.items():
        print(task, st)

build_infosearch_all()


def load_corpus(path):
    docids, texts, tasks = [], [], []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)
            docids.append(str(obj["docid"]))
            texts.append(obj["text"])
            tasks.append(obj.get("task", "unknown"))
    return docids, texts, tasks

def load_queries(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

docids, corpus_texts, corpus_tasks = load_corpus(CORPUS_PATH)
queries = load_queries(QUERIES_PATH)

print("\nLoaded normalized InfoSearch")
print("Docs:", len(docids))
print("Queries:", len(queries))
print("Queries by task:", Counter(q["task"] for q in queries))


def resolve_adapter(adapter_id_or_path, local_name):
    if os.path.exists(os.path.join(adapter_id_or_path, "adapter_config.json")):
        print("Using local adapter:", adapter_id_or_path)
        return adapter_id_or_path

    local_dir = os.path.join(ADAPTER_CACHE, local_name)

    if not os.path.exists(os.path.join(local_dir, "adapter_config.json")):
        print("Downloading adapter:", adapter_id_or_path)
        snapshot_download(
            repo_id=adapter_id_or_path,
            local_dir=local_dir,
            token=HF_TOKEN if HF_TOKEN else None,
        )

    assert os.path.exists(os.path.join(local_dir, "adapter_config.json")), local_dir
    print("Using downloaded adapter:", local_dir)
    return local_dir

ORIGINAL_ADAPTER = resolve_adapter(ORIGINAL_ADAPTER_REPO, "promptriever_original")
FINETUNED_ADAPTER = resolve_adapter(FINETUNED_ADAPTER, "promptriever_ru_finetuned")


def dcg_from_rels(rels, k):
    total = 0.0
    for i, rel in enumerate(rels[:k], start=1):
        if rel > 0:
            total += rel / math.log2(i + 1)
    return total

def ndcg_at_k(ranked_docids, relevant_docids, k):
    relset = set(relevant_docids)
    rels = [1 if d in relset else 0 for d in ranked_docids[:k]]
    dcg = dcg_from_rels(rels, k)
    ideal_rels = [1] * min(len(relset), k)
    idcg = dcg_from_rels(ideal_rels, k)
    return dcg / idcg if idcg > 0 else 0.0

def ap_at_k(ranked_docids, relevant_docids, k):
    relset = set(relevant_docids)
    if not relset:
        return 0.0
    hits = 0
    total = 0.0
    for i, d in enumerate(ranked_docids[:k], start=1):
        if d in relset:
            hits += 1
            total += hits / i
    return total / min(len(relset), k)

def recall_at_k(ranked_docids, relevant_docids, k):
    relset = set(relevant_docids)
    if not relset:
        return 0.0
    found = sum(1 for d in ranked_docids[:k] if d in relset)
    return found / len(relset)

def hit_at_k(ranked_docids, relevant_docids, k):
    relset = set(relevant_docids)
    return float(any(d in relset for d in ranked_docids[:k]))

def mrr_at_k(ranked_docids, relevant_docids, k):
    relset = set(relevant_docids)
    for i, d in enumerate(ranked_docids[:k], start=1):
        if d in relset:
            return 1.0 / i
    return 0.0

def first_rank_score_for_docs(I_row, D_row, target_docids):
    targets = set(target_docids)
    for pos, idx in enumerate(I_row):
        did = docids[int(idx)]
        if did in targets:
            return pos + 1, float(D_row[pos])
    return len(I_row) + 1, float("-inf")

def pmrr_doc_score(rank_old, rank_new):
    rr_old = 1.0 / rank_old
    rr_new = 1.0 / rank_new
    if rank_old > rank_new:
        return (rr_old / rr_new) - 1.0
    return 1.0 - (rr_new / rr_old)

def sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev):
    return float(
        (r_ins < r_ori) and
        (s_ins > s_ori) and
        (r_ori < r_rev) and
        (s_ori > s_rev)
    )

def wise_score(r_ori, r_ins, r_rev, n_positive_original=1, k=20):
    if r_ins <= r_ori < r_rev:
        if r_ori <= n_positive_original and r_ins == 1:
            return 1.0
        if r_ori <= k:
            return (1.0 - (math.sqrt(max(0, r_ori - r_ins)) / k)) * (1.0 / math.sqrt(r_ins))
        return 0.01

    if r_rev < r_ori < r_ins:
        return -1.0
    if r_ori <= r_ins:
        return (r_ori - r_ins) / r_ins
    if r_rev <= r_ori:
        return (r_rev - r_ori) / r_ori

    return 0.0

def avg(rows, key):
    return float(np.mean([r[key] for r in rows])) if rows else 0.0

def summarize_rows(rows):
    def mode_summary(mode):
        return {
            "MRR@10": avg(rows, f"mrr10_{mode}"),
            "nDCG@10": avg(rows, f"ndcg10_{mode}"),
            "MAP@1000": avg(rows, f"map1000_{mode}"),
            "Hit@10": avg(rows, f"hit10_{mode}"),
            "Recall@10": avg(rows, f"recall10_{mode}"),
            "Recall@100": avg(rows, f"recall100_{mode}"),
        }

    return {
        "count": len(rows),
        "orig": mode_summary("orig"),
        "inst": mode_summary("inst"),
        "rev": mode_summary("rev"),
        "instruction_metrics": {
            "SICR": avg(rows, "sicr"),
            "SICR_x100": 100 * avg(rows, "sicr"),
            "WISE": avg(rows, "wise"),
            "WISE_x100": 100 * avg(rows, "wise"),
            "pMRR": avg(rows, "pmrr"),
            "pMRR_x100": 100 * avg(rows, "pmrr"),
        },
    }

def query_text(q, mode):
    if mode == "orig":
        return q["query"].strip()
    if mode == "inst":
        return q["instruction_changed"].strip()
    if mode == "rev":
        return q["instruction_reversed"].strip()
    raise ValueError(mode)


def eval_adapter(label, adapter_path):
    print("\n" + "=" * 100)
    print("MODEL:", label)
    print("ADAPTER:", adapter_path)
    print("=" * 100)

    model_dir = os.path.join(WORK_DIR, label.replace("/", "_").replace(" ", "_"))
    os.makedirs(model_dir, exist_ok=True)

    emb_path = os.path.join(model_dir, "corpus_embeddings.npy")
    docids_path = os.path.join(model_dir, "docids.txt")
    per_query_out = os.path.join(model_dir, "per_query.jsonl")
    summary_out = os.path.join(model_dir, "summary.json")

    gc.collect()
    torch.cuda.empty_cache()

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )

    try:
        tokenizer = AutoTokenizer.from_pretrained(adapter_path, use_fast=False, token=HF_TOKEN if HF_TOKEN else None)
    except Exception:
        tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=False, token=HF_TOKEN if HF_TOKEN else None)

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "right"

    base = AutoModel.from_pretrained(
        BASE_MODEL,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.float16,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    base.config.use_cache = False

    model = PeftModel.from_pretrained(
        base,
        adapter_path,
        is_trainable=False,
        token=HF_TOKEN if HF_TOKEN else None,
    )
    model.eval()

    device = next(model.parameters()).device
    print("Loaded on:", device)

    def add_eos(texts):
        return [t + tokenizer.eos_token for t in texts]

    def eos_pool(last_hidden_state, attention_mask):
        lengths = attention_mask.sum(dim=1) - 1
        idx = torch.arange(last_hidden_state.size(0), device=last_hidden_state.device)
        return last_hidden_state[idx, lengths]

    @torch.inference_mode()
    def encode_texts(texts, max_len):
        batch = tokenizer(
            add_eos(texts),
            padding=True,
            truncation=True,
            max_length=max_len,
            return_tensors="pt",
        )
        batch = {k: v.to(device) for k, v in batch.items()}
        out = model(**batch)
        emb = eos_pool(out.last_hidden_state, batch["attention_mask"])
        emb = F.normalize(emb, p=2, dim=-1)
        return emb.detach().to(torch.float32).cpu().numpy()

    def encode_all(texts, max_len, batch_size, desc):
        chunks = []
        for i in tqdm(range(0, len(texts), batch_size), desc=desc):
            chunks.append(encode_texts(texts[i:i + batch_size], max_len))
        return np.vstack(chunks)

    if os.path.exists(emb_path) and os.path.exists(docids_path):
        print("Loading cached corpus embeddings:", emb_path)

        with open(docids_path, "r", encoding="utf-8") as f:
            cached_docids = [x.strip() for x in f if x.strip()]
        assert cached_docids == docids

        dim = os.path.getsize(emb_path) // (len(docids) * np.dtype(np.float16).itemsize)
        emb_mm = np.memmap(emb_path, dtype=np.float16, mode="r", shape=(len(docids), dim))
        xb = np.asarray(emb_mm, dtype=np.float32)
    else:
        print("Encoding corpus docs:", len(corpus_texts))

        first = encode_texts(corpus_texts[:1], PASSAGE_MAX_LEN).astype(np.float16)
        dim = first.shape[1]

        emb = np.memmap(emb_path, dtype=np.float16, mode="w+", shape=(len(corpus_texts), dim))
        emb[0:1] = first

        t0 = time.time()

        for i in tqdm(range(1, len(corpus_texts), BATCH_DOCS), desc="encoding corpus"):
            j = min(len(corpus_texts), i + BATCH_DOCS)
            emb[i:j] = encode_texts(corpus_texts[i:j], PASSAGE_MAX_LEN).astype(np.float16)

        emb.flush()

        with open(docids_path, "w", encoding="utf-8") as f:
            for d in docids:
                f.write(d + "\n")

        xb = np.asarray(emb, dtype=np.float32)
        print("Corpus encoded. time_s:", round(time.time() - t0, 1), "dim:", dim)

    index = faiss.IndexFlatIP(xb.shape[1])
    index.add(xb.astype(np.float32))

    print("FAISS docs:", index.ntotal)

    q_orig = encode_all([query_text(q, "orig") for q in queries], QUERY_MAX_LEN, BATCH_QUERIES, "encoding original")
    q_inst = encode_all([query_text(q, "inst") for q in queries], QUERY_MAX_LEN, BATCH_QUERIES, "encoding instructed")
    q_rev = encode_all([query_text(q, "rev") for q in queries], QUERY_MAX_LEN, BATCH_QUERIES, "encoding reversed")

    topk = len(docids)

    D_orig, I_orig = index.search(q_orig, topk)
    D_inst, I_inst = index.search(q_inst, topk)
    D_rev, I_rev = index.search(q_rev, topk)

    rows = []

    for i, q in enumerate(queries):
        ranked_orig = [docids[int(x)] for x in I_orig[i]]
        ranked_inst = [docids[int(x)] for x in I_inst[i]]
        ranked_rev = [docids[int(x)] for x in I_rev[i]]

        rel_og = q["rel_og"]
        rel_changed = q["rel_changed"]
        rel_reversed = q["rel_reversed"]
        target_docs = q["target_docs"]

        r_ori, s_ori = first_rank_score_for_docs(I_orig[i], D_orig[i], target_docs)
        r_ins, s_ins = first_rank_score_for_docs(I_inst[i], D_inst[i], target_docs)
        r_rev, s_rev = first_rank_score_for_docs(I_rev[i], D_rev[i], target_docs)

        pmrr_scores = []
        for did in q["pmrr_changed_docs"]:
            r_old, _ = first_rank_score_for_docs(I_orig[i], D_orig[i], [did])
            r_new, _ = first_rank_score_for_docs(I_inst[i], D_inst[i], [did])
            pmrr_scores.append(pmrr_doc_score(r_old, r_new))

        rows.append({
            "qid": q["qid"],
            "task": q.get("task", "unknown"),

            "rank_orig_target": r_ori,
            "score_orig_target": s_ori,
            "rank_inst_target": r_ins,
            "score_inst_target": s_ins,
            "rank_rev_target": r_rev,
            "score_rev_target": s_rev,

            "mrr10_orig": mrr_at_k(ranked_orig, rel_og, 10),
            "ndcg10_orig": ndcg_at_k(ranked_orig, rel_og, 10),
            "map1000_orig": ap_at_k(ranked_orig, rel_og, 1000),
            "hit10_orig": hit_at_k(ranked_orig, rel_og, 10),
            "recall10_orig": recall_at_k(ranked_orig, rel_og, 10),
            "recall100_orig": recall_at_k(ranked_orig, rel_og, 100),

            "mrr10_inst": mrr_at_k(ranked_inst, rel_changed, 10),
            "ndcg10_inst": ndcg_at_k(ranked_inst, rel_changed, 10),
            "map1000_inst": ap_at_k(ranked_inst, rel_changed, 1000),
            "hit10_inst": hit_at_k(ranked_inst, rel_changed, 10),
            "recall10_inst": recall_at_k(ranked_inst, rel_changed, 10),
            "recall100_inst": recall_at_k(ranked_inst, rel_changed, 100),

            "mrr10_rev": mrr_at_k(ranked_rev, rel_reversed, 10),
            "ndcg10_rev": ndcg_at_k(ranked_rev, rel_reversed, 10),
            "map1000_rev": ap_at_k(ranked_rev, rel_reversed, 1000),
            "hit10_rev": hit_at_k(ranked_rev, rel_reversed, 10),
            "recall10_rev": recall_at_k(ranked_rev, rel_reversed, 10),
            "recall100_rev": recall_at_k(ranked_rev, rel_reversed, 100),

            "sicr": sicr_score(r_ori, s_ori, r_ins, s_ins, r_rev, s_rev),
            "wise": wise_score(r_ori, r_ins, r_rev),
            "pmrr": float(np.mean(pmrr_scores)) if pmrr_scores else 0.0,
        })

    with open(per_query_out, "w", encoding="utf-8") as f:
        for r in rows:
            f.write(json.dumps(r, ensure_ascii=False) + "\n")

    summary = summarize_rows(rows)

    by_task = {}
    for task in sorted(set(r["task"] for r in rows)):
        task_rows = [r for r in rows if r["task"] == task]
        by_task[task] = summarize_rows(task_rows)

    summary["by_task"] = by_task
    summary["model"] = label
    summary["adapter"] = adapter_path
    summary["docs"] = len(docids)
    summary["queries"] = len(queries)
    summary["outputs"] = {
        "per_query": per_query_out,
        "summary": summary_out,
    }

    with open(summary_out, "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    print("\nSUMMARY:", label)
    print(json.dumps({
        "inst": summary["inst"],
        "instruction_metrics": summary["instruction_metrics"],
    }, ensure_ascii=False, indent=2))

    print("\nBY TASK:")
    for task, s in summary["by_task"].items():
        print(task, json.dumps({
            "count": s["count"],
            "inst_nDCG@10": s["inst"]["nDCG@10"],
            "inst_MRR@10": s["inst"]["MRR@10"],
            "SICR_x100": s["instruction_metrics"]["SICR_x100"],
            "WISE_x100": s["instruction_metrics"]["WISE_x100"],
            "pMRR_x100": s["instruction_metrics"]["pMRR_x100"],
        }, ensure_ascii=False))

    del model
    del base
    del tokenizer
    del index
    del q_orig
    del q_inst
    del q_rev
    gc.collect()
    torch.cuda.empty_cache()

    return summary


summaries = []

summaries.append(eval_adapter(
    "PromptRetriever-original",
    ORIGINAL_ADAPTER
))

summaries.append(eval_adapter(
    "PromptRetriever-ru-finetuned",
    FINETUNED_ADAPTER
))

comparison_path = os.path.join(WORK_DIR, "comparison_summary.json")
with open(comparison_path, "w", encoding="utf-8") as f:
    json.dump(summaries, f, ensure_ascii=False, indent=2)

print("\nSAVED:", comparison_path)


print("\nLATEX TABLE")
print(r"""
\begin{table}[H]
\centering
\small
\setlength{\tabcolsep}{5pt}
\renewcommand{\arraystretch}{1.05}
\begin{tabular}{lrrrrrrr}
\toprule
Model & MRR@10 & nDCG@10 & MAP@1000 & Hit@10 & SICR, \% & WISE, \% & p-MRR, \% \\
\midrule""")

for s in summaries:
    inst = s["inst"]
    im = s["instruction_metrics"]
    name = s["model"]

    if name == "PromptRetriever-original":
        name = "PromptRetriever"
    elif name == "PromptRetriever-ru-finetuned":
        name = "PromptRetriever + RU FT"

    print(
        f"{name} & "
        f"{inst['MRR@10']:.3f} & "
        f"{inst['nDCG@10']:.3f} & "
        f"{inst['MAP@1000']:.3f} & "
        f"{inst['Hit@10']:.3f} & "
        f"{im['SICR_x100']:.2f} & "
        f"{im['WISE_x100']:.2f} & "
        f"{im['pMRR_x100']:.2f} \\\\"
    )

print(r"""\bottomrule
\end{tabular}
\caption{Сравнение исходного PromptRetriever и модели после дообучения на русском датасете на всех подмножествах benchmark-а InfoSearch.}
\label{tab:infosearch_all_forgetting}
\end{table}
""")
